# 01 — Data preparation

**NOT RUN**

Prepared but intentionally not executed. Run manually in Google Colab.

CPU only. Supply reviewed JSONL; no dataset download occurs.

Upload the project ZIP when prompted. Set only the flags for the stage you intend to run. Model stages need `INSTALL_MODEL_PACKAGES=True`; core stages need only PyYAML. Back up datasets/checkpoints/results to Drive manually before the Colab session ends.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import zipfile

PROJECT_DIR = Path.cwd() / "certificate-recovery"
ARCHIVE = Path.cwd() / "Dependency_Certificate_Recovery_Foundation.zip"
if not PROJECT_DIR.exists():
    if not ARCHIVE.exists():
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP")
        ARCHIVE = Path.cwd() / archives[0]
    with zipfile.ZipFile(ARCHIVE) as bundle:
        destination = Path.cwd().resolve()
        for member in bundle.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)
if not (PROJECT_DIR / "src" / "cert_recovery").exists():
    raise FileNotFoundError("Project source directory missing")
sys.path.insert(0, str(PROJECT_DIR / "src"))

INSTALL_CORE_PACKAGES = False
INSTALL_MODEL_PACKAGES = False
if INSTALL_CORE_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")])
if INSTALL_MODEL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-colab.txt")])
if importlib.util.find_spec("yaml") is None:
    raise RuntimeError("Set INSTALL_CORE_PACKAGES=True and rerun setup; no model packages are needed for the core.")

from cert_recovery.config import load_config
config = load_config(PROJECT_DIR / "config" / "config.yaml")
print("Prepared but intentionally not executed. Run manually in Google Colab.")


## Inputs
The format examples are small authored examples, not research data. Use `USE_FORMAT_EXAMPLES` only for a manual setup check. Group related worlds/templates before splitting.

In [ ]:
import shutil
from cert_recovery.config import resolve_path

USE_FORMAT_EXAMPLES = False
if USE_FORMAT_EXAMPLES:
    for example, target_name in [("dependency_pairs.example.jsonl", "raw_pairs"), ("workflow_cases.example.jsonl", "workflow_cases")]:
        target = resolve_path(config, target_name)
        if target.exists():
            raise FileExistsError(f"Preserving existing input: {target}")
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(PROJECT_DIR / "data/examples" / example, target)


In [ ]:
from cert_recovery.data_pipeline import prepare_data

RUN_DATA_PREPARATION = False
if RUN_DATA_PREPARATION:
    config["execution"]["allow_data_preparation"] = True
    manifest = prepare_data(config, manual=True)
    print(manifest)
else:
    print("NOT RUN: data preparation")


Inspect class balance, grouping and labels manually before model work. Preserve split_manifest.json; do not repeatedly resplit based on test performance.